# **Hypothetical Document Embeddings (HyDE) RAG**
Uses an LLM to create a hypothetical answer or document based on the user's question.
That generated content is used to search for more relevant real documents.

### **Components & Architecture**
- **Embedding Model:** `OpenAIEmbeddings`
- **Vector Database:** Weaviate (Cloud) / Chroma (Local)
- **Hypothetical Doc Generator:** `ChatOpenAI`
- **LLM Model:** `ChatOpenAI`

## **Initial Setup**

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ['WEAVIATE_API_KEY'] = userdata.get('WEAVIATE_API_KEY')


## **Indexing**

In [ ]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# load data
from langchain.document_loaders import CSVLoader
loader = CSVLoader("./context.csv")
documents = loader.load()

In [ ]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

## **Weaviate Vector Database**

In [ ]:
# create vectorstore using weaviate
import weaviate
from weaviate.classes.init import Auth
import os

wcd_url = 'your_url'

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=wcd_url,
    auth_credentials=Auth.api_key(os.environ['WEAVIATE_API_KEY']),
    headers={'X-OpenAI-Api-key': os.environ["OPENAI_API_KEY"]}
)

In [ ]:
# create vectorstore
from langchain_weaviate.vectorstores import WeaviateVectorStore
vectorstore = WeaviateVectorStore.from_documents(documents,embedding=OpenAIEmbeddings(),client=client, index_name="your_collection_name",text_key="text")

In [ ]:
# checking similarity search
vectorstore.similarity_search("world war II", k=3)

In [ ]:
# # call vectorstore fron weaviate cloud
# vectorstore = WeaviateVectorStore(client=client,index_name="your_collection_name",embedding=OpenAIEmbeddings(),text_key="text")

## **Chromadb (Optional)**

In [ ]:
# # optional vectorstore
# !pip install chromadb
# # create vectorstore
# from langchain.vectorstores import Chroma
# vectorstore = Chroma.from_documents(documents, embeddings)

## **Retriever**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

## **Hypothetical Answer Chain**

In [ ]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# chain without the retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

template ="""
You are a helpful assistant that answers questions.
Question: {input}
Answer:
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", template),
        ("human", "{input}"),
    ]
)
qa_no_context = prompt | llm | StrOutputParser()

In [ ]:
# response
question = 'how does interlibrary loan work'
answer = qa_no_context.invoke({"input": question})
answer

## **Combined RAG Chain**

In [ ]:
# response with context
retrieval_chain = qa_no_context | retriever
retrieved_docs = retrieval_chain.invoke({"input":question})

In [ ]:
template = """
You are a helpful assistant that answers questions based on the provided context.
Use the provided context to answer the question.
Question: {input}
Context: {context}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# final response
final_rag_chain.invoke({"context":retrieved_docs,"input":question})